In [ ]:
packages <- c("readr", "dplyr", "ggplot2", "scales")

for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p)
  }
}

library(readr)
library(dplyr)
library(ggplot2)
library(scales)

# Make notebook plots comfortably large on screen
if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 14, repr.plot.height = 8, repr.plot.res = 160)
}

cat("Working directory:\n")
print(getwd())
cat("\n")

csv_path <- Sys.getenv("SIGNAL_EVENTS_CSV", unset = NA_character_)
if (is.na(csv_path) || csv_path == "") {
  event_candidates <- list.files(
    "benchmark_output",
    pattern = "^events\\.csv$",
    recursive = TRUE,
    full.names = TRUE
  )

  if (length(event_candidates) == 0) {
    stop("No events.csv found under benchmark_output. Run the Signal benchmark or set SIGNAL_EVENTS_CSV.")
  }

  csv_path <- event_candidates[which.max(file.info(event_candidates)$mtime)]
}

if (!file.exists(csv_path)) {
  stop(
    paste0(
      "CSV not found at: ", csv_path, "\n",
      "Update SIGNAL_EVENTS_CSV or run the benchmark first."
    )
  )
}

cat("Trying to read:\n")
print(csv_path)
cat("\n")

df <- read_csv(
  csv_path,
  show_col_types = FALSE,
  col_types = cols(
    worker_id = col_character(),
    ts_unix_ns = col_character()
  )
)

cat("Rows / columns:\n")
print(dim(df))
cat("\n")

cat("Operation counts:\n")
print(count(df, op))
cat("\n")

required_cols <- c("op", "member_count", "cpu_thread_ns", "artifact_size_bytes")
missing_cols <- setdiff(required_cols, names(df))
if (length(missing_cols) > 0) {
  stop(paste("Missing required columns:", paste(missing_cols, collapse = ", ")))
}

# ----------------------------
# Styling helpers
# ----------------------------
cpu_col <- "#4C78A8"
artifact_col <- "#F58518"
grid_col <- "#E6EAF0"
bg_col <- "#FBFBFD"
text_col <- "#1F2937"

pretty_theme <- function() {
  theme_minimal(base_size = 16) +
    theme(
      plot.background = element_rect(fill = bg_col, colour = NA),
      panel.background = element_rect(fill = bg_col, colour = NA),
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_line(colour = grid_col, linewidth = 0.5),
      panel.grid.major.y = element_line(colour = grid_col, linewidth = 0.6),
      plot.title = element_text(size = 22, face = "bold", colour = text_col),
      plot.subtitle = element_text(size = 13, colour = muted(text_col)),
      plot.caption = element_text(size = 11, colour = muted(text_col)),
      axis.title.x = element_text(size = 15, face = "bold", colour = text_col),
      axis.title.y.left = element_text(size = 15, face = "bold", colour = cpu_col),
      axis.title.y.right = element_text(size = 15, face = "bold", colour = artifact_col),
      axis.text = element_text(size = 12, colour = text_col),
      legend.position = "top",
      legend.direction = "horizontal",
      legend.title = element_blank(),
      legend.text = element_text(size = 12, colour = text_col),
      plot.margin = margin(15, 20, 15, 15)
    )
}

prep_dual_axis_df <- function(data, op_name) {
  data %>%
    filter(op == op_name) %>%
    filter(!is.na(member_count), !is.na(cpu_thread_ns), !is.na(artifact_size_bytes)) %>%
    mutate(
      cpu_ms = cpu_thread_ns / 1e6,
      artifact_kib = artifact_size_bytes / 1024
    ) %>%
    group_by(member_count) %>%
    summarise(
      cpu_ms = median(cpu_ms, na.rm = TRUE),
      artifact_kib = median(artifact_kib, na.rm = TRUE),
      n = n(),
      .groups = "drop"
    ) %>%
    arrange(member_count)
}

make_dual_axis_plot <- function(plot_df, title_text, subtitle_text, caption_text) {
  if (nrow(plot_df) == 0) {
    stop(paste("No usable rows for plot:", title_text))
  }

  max_cpu <- max(plot_df$cpu_ms, na.rm = TRUE)
  max_artifact <- max(plot_df$artifact_kib, na.rm = TRUE)

  if (!is.finite(max_cpu) || !is.finite(max_artifact) || max_artifact == 0) {
    scale_factor <- 1
  } else {
    scale_factor <- max_cpu / max_artifact
  }

  ggplot(plot_df, aes(x = member_count)) +
    geom_line(
      aes(y = cpu_ms, colour = "CPU thread time"),
      linewidth = 1.35,
      lineend = "round"
    ) +
    geom_point(
      aes(y = cpu_ms, colour = "CPU thread time"),
      size = 3.4,
      alpha = 0.95
    ) +
    geom_line(
      aes(y = artifact_kib * scale_factor, colour = "Artifact size"),
      linewidth = 1.35,
      lineend = "round"
    ) +
    geom_point(
      aes(y = artifact_kib * scale_factor, colour = "Artifact size"),
      size = 3.4,
      alpha = 0.95
    ) +
    scale_colour_manual(
      values = c(
        "CPU thread time" = cpu_col,
        "Artifact size" = artifact_col
      )
    ) +
    scale_x_continuous(
      breaks = pretty_breaks(n = 12),
      expand = expansion(mult = c(0.02, 0.03))
    ) +
    scale_y_continuous(
      name = "CPU thread time (ms)",
      labels = label_number(accuracy = 0.1),
      sec.axis = sec_axis(
        ~ . / scale_factor,
        name = "Artifact size (KiB)",
        labels = label_number(accuracy = 0.1)
      ),
      expand = expansion(mult = c(0.03, 0.08))
    ) +
    labs(
      title = title_text,
      subtitle = subtitle_text,
      x = "Group size (member_count)",
      caption = caption_text
    ) +
    pretty_theme()
}

# ----------------------------
# Signal operation plots
# ----------------------------
operation_specs <- data.frame(
  op = c(
    "pre_key_bundle_create",
    "add_commit_create",
    "remove_commit_create",
    "update_commit_create",
    "application_message_create",
    "application_message_receive"
  ),
  title = c(
    "Pre-key Bundle Creation: CPU Time and Artifact Size vs Group Size",
    "Pairwise Add Control: CPU Time and Artifact Size vs Group Size",
    "Pairwise Remove Control: CPU Time and Artifact Size vs Group Size",
    "Pairwise Update Control: CPU Time and Artifact Size vs Group Size",
    "Pairwise Application Send: CPU Time and Artifact Size vs Group Size",
    "Pairwise Application Receive: CPU Time and Artifact Size vs Group Size"
  ),
  subtitle = c(
    "X3DH public bundle generation; group size reflects the worker's current local state",
    "One group-control envelope with per-recipient X3DH or session ciphertexts",
    "One group-control envelope with per-recipient session ciphertexts for remaining members",
    "Self-update broadcast with one encrypted control message per peer",
    "Native Signal baseline: one ciphertext per recipient; payload sizes are collapsed to medians",
    "Sampled recipient-side decryptions; points are medians per group size"
  ),
  file = c(
    "pre_key_bundle_create_cpu_and_artifact_vs_group_size.png",
    "add_commit_create_cpu_and_artifact_vs_group_size.png",
    "remove_commit_create_cpu_and_artifact_vs_group_size.png",
    "update_commit_create_cpu_and_artifact_vs_group_size.png",
    "application_message_create_cpu_and_artifact_vs_group_size.png",
    "application_message_receive_cpu_and_artifact_vs_group_size.png"
  ),
  stringsAsFactors = FALSE
) %>%
  filter(op %in% unique(df$op))

if (nrow(operation_specs) == 0) {
  stop("No known Signal benchmark operations were found in the CSV.")
}

plot_objects <- list()
plot_files <- character()
caption_text <- "Left axis: CPU thread time (ms). Right axis: serialized artifact size (KiB)."

for (i in seq_len(nrow(operation_specs))) {
  spec <- operation_specs[i, ]
  summary_df <- prep_dual_axis_df(df, spec$op)

  cat("Prepared ", spec$op, " data:\n", sep = "")
  print(summary_df)
  cat("\n")

  plot_objects[[spec$op]] <- make_dual_axis_plot(
    summary_df,
    title_text = spec$title,
    subtitle_text = spec$subtitle,
    caption_text = caption_text
  )
  plot_files[[spec$op]] <- spec$file
}

for (plot_name in names(plot_objects)) {
  print(plot_objects[[plot_name]])
}

# ----------------------------
# Save high-resolution files for thesis use
# ----------------------------
out_dir <- file.path(dirname(csv_path), "plots")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

for (plot_name in names(plot_objects)) {
  ggsave(
    filename = file.path(out_dir, plot_files[[plot_name]]),
    plot = plot_objects[[plot_name]],
    width = 14,
    height = 8,
    dpi = 320,
    bg = bg_col
  )
}

cat("Saved plots to:\n")
print(out_dir)
